In [58]:
import pandas as pd

train_x = pd.read_csv("train.csv")
test_x = pd.read_csv("test.csv")

In [59]:
def preprocessing_Timestamp(df):
    # Convert TIMESTAMP to datetime
    df["TIMESTAMP"] = pd.to_datetime(df["TIMESTAMP"])
    
    # Extract date components
    df["Month"] = df["TIMESTAMP"].dt.month
    df["Day"] = df["TIMESTAMP"].dt.day
    df["Hour"] = df["TIMESTAMP"].dt.hour
    df["Minute"] = df["TIMESTAMP"].dt.minute

    df = df.drop(["TIMESTAMP"], axis = 1) # TIMESTAMP 열 삭제
    
    return df

train_x = preprocessing_Timestamp(train_x)
test_x = preprocessing_Timestamp(test_x)

In [60]:
train_x = train_x.drop(columns=['PRODUCT_ID', 'Y_Quality'])
train_y = train_x['Y_Class']

test_x = test_x.drop(columns=['PRODUCT_ID'])

In [61]:
from sklearn.preprocessing import LabelEncoder

def preprocessing_Line_Product(df):
    # 'LINE'과 'PRODUCT_CODE_encoded'의 조합을 하나의 문자열로 결합
    df['LINE_PRODUCT_COMBINATION'] = df['LINE'].astype(str) + '_' + df['PRODUCT_CODE'].astype(str)

    # LabelEncoder를 사용하여 고유한 숫자 레이블 부여
    label_encoder = LabelEncoder()
    df['LINE_PRODUCT_LABEL'] = label_encoder.fit_transform(df['LINE_PRODUCT_COMBINATION'])

    df = df.drop(["LINE", "PRODUCT_CODE", "LINE_PRODUCT_COMBINATION"], axis = 1)

    return df

train_x = preprocessing_Line_Product(train_x)
test_x = preprocessing_Line_Product(test_x)

In [62]:
from sklearn.preprocessing import MinMaxScaler

# X 컬럼 추출
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 그룹 매핑 함수
def map_line_group(label):
    if label in [0, 1]:
        return 0
    elif label in [2, 3]:
        return 1
    elif label in [4, 6]:
        return 2
    elif label in [5, 7]:
        return 3
    else:
        return -1

# 학습 데이터에 그룹 라벨 추가
train_x['LINE_GROUP_LABEL'] = train_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 스케일러 저장 딕셔너리
group_scalers = {}

# 그룹별 정규화 및 스케일러 저장
for group_label in train_x['LINE_GROUP_LABEL'].unique():
    idx = train_x['LINE_GROUP_LABEL'] == group_label
    scaler = MinMaxScaler()
    train_x.loc[idx, x_cols] = scaler.fit_transform(train_x.loc[idx, x_cols])
    group_scalers[group_label] = scaler  # 저장

# 보조 컬럼 제거
train_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

# 테스트 데이터에도 그룹 라벨 생성
test_x['LINE_GROUP_LABEL'] = test_x['LINE_PRODUCT_LABEL'].apply(map_line_group)

# 그룹별 정규화 적용
for group_label in test_x['LINE_GROUP_LABEL'].unique():
    idx = test_x['LINE_GROUP_LABEL'] == group_label
    scaler = group_scalers[group_label]  # 학습된 스케일러 가져오기
    test_x.loc[idx, x_cols] = scaler.transform(test_x.loc[idx, x_cols])

# 필요 시 제거
test_x.drop(columns=['LINE_GROUP_LABEL'], inplace=True)

c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:793: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmax(X, axis=axis))
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\_array_api.py:776: RuntimeWarning: All-NaN slice encountered
  return xp.asarray(numpy.nanmin(X, axis=axis))
c:\Users\twoh0\AppDa

In [63]:
#빈 slot을 모두 평균으로 채움
train_x = train_x.fillna(train_x.mean())
test_x = test_x.fillna(train_x.mean())

train_x = train_x.dropna(axis = 1)
test_x = test_x.dropna(axis = 1)

train_x.head()

,Y_Class,X_1,X_2,X_3,X_4,X_5,X_6,X_7,X_8,X_9,...,X_2867,X_2868,X_2869,X_2870,X_2871,Month,Day,Hour,Minute,LINE_PRODUCT_LABEL
0,1,0.016251,0.542605,0.0,0.0,0.39255,0.0,0.223959,0.048711,0.498567,...,0.248647,0.000000,0.122283,0.890487,0.0,6,13,5,14,2
1,2,0.016251,0.542605,0.0,0.0,0.39255,0.0,0.223959,0.048711,0.498567,...,0.300866,0.407899,0.164742,0.601770,0.0,6,13,5,22,3
2,1,0.016251,0.542605,0.0,0.0,0.39255,0.0,0.223959,0.048711,0.498567,...,0.133929,0.355835,0.205163,0.922566,0.0,6,13,5,30,2
3,2,0.016251,0.542605,0.0,0.0,0.39255,0.0,0.223959,0.048711,0.498567,...,0.202110,0.704129,0.003057,0.559181,0.0,6,13,5,39,3
4,1,0.016251,0.542605,0.0,0.0,0.39255,0.0,0.223959,0.048711,0.498567,...,0.275703,0.515978,0.088315,0.846239,0.0,6,13,5,47,2


In [64]:
# X로 시작하는 수치형 컬럼 선택
x_cols = train_x.columns[train_x.columns.str.startswith('X')].tolist()

# 상관관계 계산
corr = train_x[x_cols + ['Y_Class']].corr()

# Y_Class 기준 상관계수 정렬 (상위 몇 개만 보기 원할 수도 있음)
corr_with_target = corr['Y_Class'].drop('Y_Class').sort_values(key = abs, ascending=False)

top_features = [i for i in corr_with_target.index if abs(corr_with_target[i]) > 0.05]
top_features_sorted = [col for col in train_x.columns if col in top_features]

train_x = train_x[["Y_Class"] + top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']]
test_x = test_x[top_features_sorted + ['Month', 'Day', 'Hour', 'Minute', 'LINE_PRODUCT_LABEL']] 

train_x
# test_x

,Y_Class,X_2,X_5,X_8,X_24,X_38,X_44,X_56,X_62,X_73,...,X_2865,X_2866,X_2867,X_2869,X_2870,Month,Day,Hour,Minute,LINE_PRODUCT_LABEL
0,1,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.550000,0.256757,0.248647,0.122283,0.890487,6,13,5,14,2
1,2,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.550000,0.240754,0.300866,0.164742,0.601770,6,13,5,22,3
2,1,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.550000,0.251422,0.133929,0.205163,0.922566,6,13,5,30,2
3,2,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.550000,0.199858,0.202110,0.003057,0.559181,6,13,5,39,3
4,1,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.500000,0.233997,0.275703,0.088315,0.846239,6,13,5,47,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,1,0.533333,0.00000,0.000000,0.00000,0.000000,0.530612,0.644444,0.570093,0.641975,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,8,14,30,7
594,0,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.550000,0.616999,0.578193,0.835938,0.266593,9,8,22,38,2
595,0,0.542605,0.39255,0.048711,0.13467,0.590974,0.406615,0.505571,0.497577,0.507610,...,0.500000,0.664555,0.592741,0.719083,0.275426,9,8,22,47,2
596,1,0.538462,1.00000,0.000000,0.00000,0.750000,0.173913,0.000000,0.131148,1.000000,...,0.589759,0.664555,0.592741,0.719083,0.275426,9,8,14,38,4


In [76]:
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import make_classification

label1 = [0,1]
label2 = [2,3]
label3 = [4,6]
label4 = [5,7]

label1_df = train_x[train_x['LINE_PRODUCT_LABEL'].isin(label1)]
label2_df = train_x[train_x['LINE_PRODUCT_LABEL'].isin(label2)]
label3_df = train_x[train_x['LINE_PRODUCT_LABEL'].isin(label3)]
label4_df = train_x[train_x['LINE_PRODUCT_LABEL'].isin(label4)]

label1_df_x = label1_df.drop(["Y_Class"], axis = 1)
label2_df_x = label2_df.drop(["Y_Class"], axis = 1)
label3_df_x = label3_df.drop(["Y_Class"], axis = 1)
label4_df_x = label4_df.drop(["Y_Class"], axis = 1)

label4_df["Y_Class"].value_counts()

# model1 = LGBMClassifier(n_estimators=100, random_state=100)  # label
# model1.fit(label1_df_x, label1_df['Y_Class'])

# model2 = XGBClassifier(n_estimators=100, scale_pos_weight=1.5, use_label_encoder=False, eval_metric='mlogloss')  # label2
# model2.fit(label2_df_x, label2_df['Y_Class'])

# model3 = KNeighborsClassifier(n_neighbors=1) 
# model3.fit(label3_df_x, label3_df['Y_Class'])

# model4 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=100)
# model4.fit(label4_df_x, label4_df['Y_Class'])

Y_Class
1    285
2     30
0     28
Name: count, dtype: int64

In [70]:
def predict_label(dfs):
    Y_label = []
    
    for i, j in dfs.iterrows():
        if j['LINE_PRODUCT_LABEL'] in label1:
            Y_label.append(model1.predict([j])[0])
        elif j['LINE_PRODUCT_LABEL'] in label2:
            Y_label.append(model2.predict([j])[0])
        elif j['LINE_PRODUCT_LABEL'] in label3:
            Y_label.append(model3.predict([j])[0])
        else:
            Y_label.append(model4.predict([j])[0])
    return Y_label

print(predict_label(test_x))

c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\twoh0\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\uti

[1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 0, 2, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
